# Artists vs. Diffusion Models: Experiments

Companion notebook for the blog post at https://glopezas.github.io/diffusion-art/

**Run order:** Cells must be run top-to-bottom. Each section unloads previous models before loading new ones.

**Hardware:** T4 GPU (free Colab tier). Estimated total runtime: 2-3 hours.

---
## Cell 1: Colab Setup

In [ ]:
# Colab Setup
import os
import sys

ON_COLAB = 'google.colab' in str(get_ipython())

if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PATH = "/content/drive/Othercomputers/laptop/research/diffusion_art"
else:
    possible_paths = [
        "/mnt/c/research/diffusion_art",
        os.path.expanduser("~/research/diffusion_art"),
        ".",
    ]
    PATH = None
    for p in possible_paths:
        if os.path.exists(os.path.join(p, "assets")):
            PATH = os.path.abspath(p)
            break
    if PATH is None:
        raise RuntimeError("Could not find diffusion_art folder. Set PATH manually.")

print(f"ON_COLAB: {ON_COLAB}")
print(f"PATH: {PATH}")

os.chdir(PATH)
sys.path.insert(0, PATH)

---
## Cell 2: Install Dependencies

In [ ]:
# Install dependencies
!pip install -q diffusers==0.27.2 transformers==4.40.0 accelerate
!pip install -q clip-retrieval ftfy regex
!pip install -q plotly scikit-learn
!pip install -q open_clip_torch

# CSD repo for style similarity
if not os.path.exists("csd_repo"):
    !git clone -q https://github.com/learn2phoenix/CSD.git csd_repo
sys.path.insert(0, os.path.join(PATH, "csd_repo"))

print("Dependencies installed.")

---
## Cell 3: Global Imports and Config

In [ ]:
import os, sys, json, glob
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
from sklearn.metrics import roc_curve, auc

# Directories
ASSETS_DIR   = os.path.join(PATH, "assets")
PORTFOLIO_DIR = os.path.join(ASSETS_DIR, "portfolio")
CONTROL_DIR  = os.path.join(ASSETS_DIR, "control")
GENERATED_DIR = os.path.join(ASSETS_DIR, "generated")
LAION_DIR    = os.path.join(ASSETS_DIR, "laion_matches")
GEMINI_DIR   = os.path.join(ASSETS_DIR, "gemini")
FIGURES_DIR  = os.path.join(ASSETS_DIR, "figures")
RESULTS_DIR  = os.path.join(PATH, "results")

for d in [GENERATED_DIR, LAION_DIR, GEMINI_DIR, FIGURES_DIR, RESULTS_DIR,
          os.path.join(GENERATED_DIR, "sd15_style"),
          os.path.join(GENERATED_DIR, "sd21_style"),
          os.path.join(GENERATED_DIR, "sd15_caption"),
          os.path.join(GENERATED_DIR, "mia")]:
    os.makedirs(d, exist_ok=True)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Results accumulator
results = {}

# Plotly template
pio.templates.default = "plotly_white"

---
## Cell 4: Load Artist Portfolios

**Before running:** Add images to:
- `assets/portfolio/` — 10-15 Rutkowski paintings (PNG or JPG)
- `assets/control/` — 10-15 Charlie Bowater paintings (PNG or JPG)

Good sources: ArtStation (artstation.com/rutkowski, artstation.com/charliebowater)

In [ ]:
def load_images(directory, size=(512, 512)):
    """Load all images from a directory, return as list of PIL Images."""
    paths = sorted(glob.glob(os.path.join(directory, "*.png")) +
                   glob.glob(os.path.join(directory, "*.jpg")) +
                   glob.glob(os.path.join(directory, "*.jpeg")))
    images = []
    for p in paths:
        img = Image.open(p).convert("RGB")
        if size:
            img = img.resize(size, Image.LANCZOS)
        images.append({"path": p, "name": os.path.basename(p), "image": img})
    print(f"  Loaded {len(images)} images from {directory}")
    return images

print("Loading Rutkowski portfolio...")
rutkowski_imgs = load_images(PORTFOLIO_DIR)

print("Loading Bowater (control) portfolio...")
bowater_imgs = load_images(CONTROL_DIR)

if not rutkowski_imgs:
    print("WARNING: No Rutkowski images found. Add images to assets/portfolio/")
if not bowater_imgs:
    print("WARNING: No Bowater images found. Add images to assets/control/")

---
## Section 2: LAION CLIP Retrieval

In [ ]:
# sec2_retrieve_and_visualize
from clip_retrieval.clip_client import ClipClient, Modality

client = ClipClient(
    url="https://knn.laion.ai/knn-service",
    indice_name="laion5B-L-14",  # LAION-5B index with ViT-L/14 embeddings
    num_images=5,
    aesthetic_score=0,
    aesthetic_weight=0,
)

def retrieve_laion_matches(images, client, save_dir):
    """Query LAION knn for each image, return results list."""
    all_results = []
    for img_data in images:
        img = img_data["image"]
        name = img_data["name"]
        print(f"  Querying: {name}")
        try:
            results_raw = client.query(image=img)
            # results_raw is a list of dicts with keys: url, caption, similarity, id
            top = results_raw[:5] if results_raw else []
            all_results.append({
                "query_name": name,
                "query_image": img,
                "matches": top,
            })
        except Exception as e:
            print(f"    Error: {e}")
            all_results.append({"query_name": name, "query_image": img, "matches": []})
    return all_results

print("Retrieving Rutkowski matches from LAION...")
rutkowski_laion = retrieve_laion_matches(rutkowski_imgs, client, LAION_DIR)

print("Retrieving Bowater matches from LAION...")
bowater_laion = retrieve_laion_matches(bowater_imgs, client, LAION_DIR)

# Save raw results
import pickle
with open(os.path.join(RESULTS_DIR, "laion_results.pkl"), "wb") as f:
    pickle.dump({"rutkowski": rutkowski_laion, "bowater": bowater_laion}, f)

print("Done.")

In [ ]:
# sec2_gallery
# Figure 2a: Side-by-side gallery of Rutkowski images vs. LAION nearest neighbor
import requests
from io import BytesIO

def fetch_image_from_url(url, timeout=5):
    try:
        r = requests.get(url, timeout=timeout)
        return Image.open(BytesIO(r.content)).convert("RGB")
    except:
        return None

# Build gallery: top-3 Rutkowski images with their best LAION match
gallery_items = [(r, r["matches"][0]) for r in rutkowski_laion if r["matches"]]
gallery_items.sort(key=lambda x: x[1]["similarity"], reverse=True)
gallery_items = gallery_items[:5]

n = len(gallery_items)
if n > 0:
    fig, axes = plt.subplots(n, 2, figsize=(10, n * 3.5))
    if n == 1:
        axes = [axes]
    fig.suptitle("Rutkowski Portfolio vs. Nearest LAION Match", fontsize=14, fontweight="bold", y=1.01)

    for i, (query, match) in enumerate(gallery_items):
        # Query image
        axes[i][0].imshow(query["query_image"])
        axes[i][0].set_title(f"Original: {query['query_name']}", fontsize=9)
        axes[i][0].axis("off")

        # LAION match
        laion_img = fetch_image_from_url(match["url"])
        if laion_img:
            axes[i][1].imshow(laion_img)
            axes[i][1].set_title(
                f"LAION match | sim = {match['similarity']:.3f}\n{match.get('caption', '')[:60]}",
                fontsize=8
            )
        else:
            axes[i][1].text(0.5, 0.5, "[URL expired]", ha="center", va="center", fontsize=10)
            axes[i][1].set_title(f"sim = {match['similarity']:.3f}", fontsize=9)
        axes[i][1].axis("off")

    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, "fig2a_laion_gallery.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: fig2a_laion_gallery.png")

In [ ]:
# sec2_comparison_chart
# Figure 2b: Number of LAION hits above threshold for each artist

THRESHOLD = 0.90

def count_hits(laion_results, threshold):
    hits = []
    for item in laion_results:
        sims = [m["similarity"] for m in item["matches"]]
        top_sim = max(sims) if sims else 0.0
        hits.append(top_sim)
    return hits

rutkowski_sims = count_hits(rutkowski_laion, THRESHOLD)
bowater_sims   = count_hits(bowater_laion, THRESHOLD)

# Store results
results["sec2"] = {
    "rutkowski_top_similarities": rutkowski_sims,
    "bowater_top_similarities": bowater_sims,
    "rutkowski_hits_above_090": sum(s >= THRESHOLD for s in rutkowski_sims),
    "bowater_hits_above_090": sum(s >= THRESHOLD for s in bowater_sims),
    "threshold": THRESHOLD,
}

# Plotly bar chart
fig = go.Figure()

fig.add_trace(go.Bar(
    name="Greg Rutkowski",
    x=[f"img_{i+1}" for i in range(len(rutkowski_sims))],
    y=rutkowski_sims,
    marker_color="#dc2626",
    opacity=0.85,
))

fig.add_trace(go.Bar(
    name="Charlie Bowater (control)",
    x=[f"img_{i+1}" for i in range(len(bowater_sims))],
    y=bowater_sims,
    marker_color="#2563eb",
    opacity=0.85,
))

fig.add_hline(y=THRESHOLD, line_dash="dash", line_color="#94a3b8",
              annotation_text=f"Threshold = {THRESHOLD}")

fig.update_layout(
    title="LAION-5B: Top CLIP Cosine Similarity per Portfolio Image",
    xaxis_title="Portfolio image",
    yaxis_title="Max cosine similarity to LAION neighbor",
    yaxis=dict(range=[0, 1]),
    barmode="group",
    legend=dict(orientation="h", y=-0.15),
    font=dict(family="system-ui, sans-serif", size=12),
)

pio.write_json(fig, os.path.join(FIGURES_DIR, "fig2b_laion_hits.json"))
fig.write_image(os.path.join(FIGURES_DIR, "fig2b_laion_hits.png"), scale=2)
fig.show()
print(f"Rutkowski hits (sim > {THRESHOLD}): {results['sec2']['rutkowski_hits_above_090']} / {len(rutkowski_sims)}")
print(f"Bowater hits (sim > {THRESHOLD}): {results['sec2']['bowater_hits_above_090']} / {len(bowater_sims)}")

---
## Section 3a: Loss-Based Membership Inference

Loads SD 1.5. Computes denoising MSE at multiple timesteps for each image.

In [ ]:
# Load SD 1.5 (VAE + UNet only; no full pipeline for memory efficiency)
from diffusers import StableDiffusionPipeline, DDPMScheduler

MODEL_ID_15 = "runwayml/stable-diffusion-v1-5"

print("Loading SD 1.5...")
pipe_15 = StableDiffusionPipeline.from_pretrained(
    MODEL_ID_15,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False,
).to(device)
pipe_15.enable_attention_slicing()

mia_scheduler = DDPMScheduler.from_pretrained(MODEL_ID_15, subfolder="scheduler")

print("SD 1.5 loaded.")

In [ ]:
# sec3a_mia_compute
from torchvision import transforms

img_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),  # [-1, 1]
])

MIA_TIMESTEPS = [50, 100, 200, 400, 600, 800]

def compute_mia_loss(pil_image, pipe, scheduler, timesteps, device):
    img_t = img_transform(pil_image).unsqueeze(0).to(device, dtype=torch.float16)

    with torch.no_grad():
        latent = pipe.vae.encode(img_t).latent_dist.mean
        latent = latent * pipe.vae.config.scaling_factor

    # Null text conditioning (unconditional)
    with torch.no_grad():
        null_ids = pipe.tokenizer("", return_tensors="pt",
                                   padding="max_length",
                                   max_length=pipe.tokenizer.model_max_length,
                                   truncation=True).input_ids.to(device)
        null_emb = pipe.text_encoder(null_ids).last_hidden_state  # [1, 77, 768]

    losses = []
    noise = torch.randn_like(latent)

    for t_val in timesteps:
        t_tensor = torch.tensor([t_val], device=device, dtype=torch.long)
        noisy = scheduler.add_noise(latent.float(), noise.float(), t_tensor).half()

        with torch.no_grad():
            pred = pipe.unet(noisy, t_tensor, encoder_hidden_states=null_emb).sample

        mse = F.mse_loss(pred.float(), noise.float()).item()
        losses.append(mse)

    return losses  # one value per timestep

print("Running MIA for Rutkowski portfolio...")
rutkowski_mia = []
for item in rutkowski_imgs:
    losses = compute_mia_loss(item["image"], pipe_15, mia_scheduler, MIA_TIMESTEPS, device)
    rutkowski_mia.append(losses)
    print(f"  {item['name']}: mean loss = {np.mean(losses):.4f}")

print("Running MIA for Bowater portfolio...")
bowater_mia = []
for item in bowater_imgs:
    losses = compute_mia_loss(item["image"], pipe_15, mia_scheduler, MIA_TIMESTEPS, device)
    bowater_mia.append(losses)
    print(f"  {item['name']}: mean loss = {np.mean(losses):.4f}")

rutkowski_mia = np.array(rutkowski_mia)  # [N, len(timesteps)]
bowater_mia   = np.array(bowater_mia)

In [ ]:
# sec3a_loss_curves
# Figure 3a: Loss curves per group

def mean_std(arr):
    return arr.mean(axis=0), arr.std(axis=0)

r_mean, r_std = mean_std(rutkowski_mia)
b_mean, b_std = mean_std(bowater_mia)

fig = go.Figure()

# Rutkowski
fig.add_trace(go.Scatter(
    x=MIA_TIMESTEPS, y=r_mean, mode="lines+markers",
    name="Rutkowski (LAION member)",
    line=dict(color="#dc2626", width=2.5),
))
fig.add_trace(go.Scatter(
    x=MIA_TIMESTEPS + MIA_TIMESTEPS[::-1],
    y=list(r_mean + r_std) + list((r_mean - r_std)[::-1]),
    fill="toself", fillcolor="rgba(220,38,38,0.12)",
    line=dict(width=0), showlegend=False, name=""
))

# Bowater
fig.add_trace(go.Scatter(
    x=MIA_TIMESTEPS, y=b_mean, mode="lines+markers",
    name="Bowater (control)",
    line=dict(color="#2563eb", width=2.5),
))
fig.add_trace(go.Scatter(
    x=MIA_TIMESTEPS + MIA_TIMESTEPS[::-1],
    y=list(b_mean + b_std) + list((b_mean - b_std)[::-1]),
    fill="toself", fillcolor="rgba(37,99,235,0.12)",
    line=dict(width=0), showlegend=False, name=""
))

fig.update_layout(
    title="Figure 3a: Denoising Loss vs. Timestep (MIA)",
    xaxis_title="Diffusion timestep t",
    yaxis_title="Mean prediction MSE",
    legend=dict(orientation="h", y=-0.15),
    font=dict(family="system-ui, sans-serif", size=12),
)

pio.write_json(fig, os.path.join(FIGURES_DIR, "fig3a_mia_loss_curves.json"))
fig.write_image(os.path.join(FIGURES_DIR, "fig3a_mia_loss_curves.png"), scale=2)
fig.show()

In [ ]:
# sec3a_score_distributions
# Figure 3b: Membership score histogram

rutkowski_scores = rutkowski_mia.mean(axis=1)
bowater_scores   = bowater_mia.mean(axis=1)

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=rutkowski_scores, name="Rutkowski",
    opacity=0.65, marker_color="#dc2626", nbinsx=10,
))
fig.add_trace(go.Histogram(
    x=bowater_scores, name="Bowater (control)",
    opacity=0.65, marker_color="#2563eb", nbinsx=10,
))

fig.update_layout(
    barmode="overlay",
    title="Figure 3b: Membership Score Distribution",
    xaxis_title="Mean denoising MSE (lower = more likely member)",
    yaxis_title="Count",
    legend=dict(orientation="h", y=-0.15),
    font=dict(family="system-ui, sans-serif", size=12),
)

pio.write_json(fig, os.path.join(FIGURES_DIR, "fig3b_mia_scores.json"))
fig.write_image(os.path.join(FIGURES_DIR, "fig3b_mia_scores.png"), scale=2)
fig.show()

In [ ]:
# sec3a_roc
# Figure 3c: ROC curve
# Label: Rutkowski = 1 (member), Bowater = 0 (non-member)
# Score: lower loss = more likely member, so use negative loss as score

scores = np.concatenate([-rutkowski_scores, -bowater_scores])
labels = np.concatenate([np.ones(len(rutkowski_scores)), np.zeros(len(bowater_scores))])

fpr, tpr, _ = roc_curve(labels, scores)
roc_auc = auc(fpr, tpr)

results["sec3a"] = {
    "rutkowski_mean_loss": float(rutkowski_scores.mean()),
    "bowater_mean_loss": float(bowater_scores.mean()),
    "mia_auc": float(roc_auc),
}

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fpr, y=tpr, mode="lines",
    name=f"Loss-based MIA (AUC = {roc_auc:.3f})",
    line=dict(color="#7c3aed", width=2.5),
))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines",
    name="Chance (AUC = 0.50)",
    line=dict(color="#94a3b8", width=1.5, dash="dash"),
))

fig.update_layout(
    title=f"Figure 3c: ROC Curve for Loss-Based Membership Inference",
    xaxis_title="False positive rate",
    yaxis_title="True positive rate",
    legend=dict(orientation="h", y=-0.15),
    font=dict(family="system-ui, sans-serif", size=12),
    xaxis=dict(range=[0, 1]),
    yaxis=dict(range=[0, 1]),
)

pio.write_json(fig, os.path.join(FIGURES_DIR, "fig3c_mia_roc.json"))
fig.write_image(os.path.join(FIGURES_DIR, "fig3c_mia_roc.png"), scale=2)
fig.show()
print(f"MIA AUC: {roc_auc:.3f}")

---
## Section 3b: Caption-Conditioned Generation

In [ ]:
# sec3b_generate
# Generate images from LAION captions (Rutkowski) and BLIP-2 captions (Bowater)

from diffusers import StableDiffusionPipeline

# SD 1.5 should still be loaded from Section 3a
# pipe_15 is available

SEEDS_CAPTION = [42, 1337, 999]
NUM_STEPS = 30
GUIDANCE = 7.5

def generate_from_caption(pipe, caption, seeds, save_dir, prefix):
    """Generate images from a text caption with multiple seeds."""
    generated = []
    for seed in seeds:
        generator = torch.Generator(device=device).manual_seed(seed)
        with torch.autocast("cuda"):
            img = pipe(
                caption,
                num_inference_steps=NUM_STEPS,
                guidance_scale=GUIDANCE,
                generator=generator,
            ).images[0]
        fname = f"{prefix}_seed{seed}.png"
        img.save(os.path.join(save_dir, fname))
        generated.append({"image": img, "seed": seed, "path": fname})
    return generated

# Retrieve LAION captions for Rutkowski images that had matches
rutkowski_captions = []
for item in rutkowski_laion:
    if item["matches"]:
        cap = item["matches"][0].get("caption", "")
        rutkowski_captions.append({"name": item["query_name"],
                                   "query_image": item["query_image"],
                                   "caption": cap})

# For Bowater, generate captions with BLIP-2
from transformers import Blip2Processor, Blip2ForConditionalGeneration

print("Loading BLIP-2 for Bowater caption generation...")
blip_processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
blip_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b", torch_dtype=torch.float16
).to(device)

def generate_blip_caption(pil_image, processor, model, device):
    inputs = processor(images=pil_image, return_tensors="pt").to(device, torch.float16)
    generated_ids = model.generate(**inputs, max_new_tokens=60)
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

bowater_captions = []
for item in bowater_imgs[:5]:  # Caption first 5 for demo
    cap = generate_blip_caption(item["image"], blip_processor, blip_model, device)
    bowater_captions.append({"name": item["name"], "query_image": item["image"], "caption": cap})
    print(f"  {item['name']}: {cap}")

# Free BLIP-2 before generation
del blip_model, blip_processor
torch.cuda.empty_cache()

# Generate images from captions
save_dir_cap = os.path.join(GENERATED_DIR, "sd15_caption")

print("Generating from Rutkowski LAION captions...")
rutkowski_caption_gen = []
for i, item in enumerate(rutkowski_captions[:3]):  # First 3 for demo
    gen = generate_from_caption(pipe_15, item["caption"], SEEDS_CAPTION,
                                 save_dir_cap, f"rutkowski_cap{i}")
    rutkowski_caption_gen.append({**item, "generated": gen})
    print(f"  Generated 3 images for: {item['caption'][:50]}")

print("Generating from Bowater BLIP-2 captions...")
bowater_caption_gen = []
for i, item in enumerate(bowater_captions[:3]):
    gen = generate_from_caption(pipe_15, item["caption"], SEEDS_CAPTION,
                                 save_dir_cap, f"bowater_cap{i}")
    bowater_caption_gen.append({**item, "generated": gen})
    print(f"  Generated 3 images for: {item['caption'][:50]}")

In [ ]:
# sec3b_clip_similarity
# Compute CLIP ViT-L/14 similarity between generated images and originals

import open_clip

print("Loading CLIP ViT-L/14...")
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-L-14", pretrained="openai"
)
clip_model = clip_model.to(device).eval()

def clip_embed(pil_image, model, preprocess, device):
    img_t = preprocess(pil_image).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = model.encode_image(img_t)
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.cpu().float()

def clip_similarity(img1, img2, model, preprocess, device):
    e1 = clip_embed(img1, model, preprocess, device)
    e2 = clip_embed(img2, model, preprocess, device)
    return (e1 * e2).sum().item()

# Compute similarities
rutkowski_cap_sims = []
for item in rutkowski_caption_gen:
    for gen in item["generated"]:
        sim = clip_similarity(item["query_image"], gen["image"], clip_model, clip_preprocess, device)
        rutkowski_cap_sims.append(sim)

bowater_cap_sims = []
for item in bowater_caption_gen:
    for gen in item["generated"]:
        sim = clip_similarity(item["query_image"], gen["image"], clip_model, clip_preprocess, device)
        bowater_cap_sims.append(sim)

# Random baseline: pair Rutkowski images with Bowater generated images
baseline_sims = []
for item in rutkowski_caption_gen:
    for b_item in bowater_caption_gen:
        for gen in b_item["generated"]:
            sim = clip_similarity(item["query_image"], gen["image"],
                                   clip_model, clip_preprocess, device)
            baseline_sims.append(sim)
baseline_sims = baseline_sims[:len(rutkowski_cap_sims)]  # Match length

results["sec3b"] = {
    "rutkowski_clip_mean": float(np.mean(rutkowski_cap_sims)),
    "bowater_clip_mean": float(np.mean(bowater_cap_sims)),
    "baseline_clip_mean": float(np.mean(baseline_sims)),
}

print(f"Rutkowski CLIP sim: {np.mean(rutkowski_cap_sims):.3f}")
print(f"Bowater CLIP sim:   {np.mean(bowater_cap_sims):.3f}")
print(f"Baseline CLIP sim:  {np.mean(baseline_sims):.3f}")

In [ ]:
# sec3b_gallery
# Figure 3d: Original + caption + 3 generated images

n = min(3, len(rutkowski_caption_gen))
if n > 0:
    fig, axes = plt.subplots(n, 4, figsize=(14, n * 3.5))
    if n == 1:
        axes = [axes]
    fig.suptitle("Original vs. Caption-Conditioned Generations (SD 1.5)",
                  fontsize=13, fontweight="bold", y=1.01)

    for row, item in enumerate(rutkowski_caption_gen[:n]):
        axes[row][0].imshow(item["query_image"])
        axes[row][0].set_title("Original", fontsize=9, fontweight="bold")
        axes[row][0].axis("off")

        caption_short = item["caption"][:60] + "..." if len(item["caption"]) > 60 else item["caption"]
        for col, gen in enumerate(item["generated"][:3]):
            sims_for_gen = clip_similarity(item["query_image"], gen["image"],
                                           clip_model, clip_preprocess, device)
            axes[row][col + 1].imshow(gen["image"])
            axes[row][col + 1].set_title(f"seed {gen['seed']}\nCLIP={sims_for_gen:.3f}",
                                          fontsize=8)
            axes[row][col + 1].axis("off")

        axes[row][0].set_xlabel(f'"{caption_short}"', fontsize=7, labelpad=4)

    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, "fig3d_caption_gallery.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: fig3d_caption_gallery.png")

In [ ]:
# sec3b_clip_distributions
# Figure 3e: CLIP similarity violin plot

import plotly.graph_objects as go

fig = go.Figure()

for label, data, color in [
    ("Rutkowski (real LAION captions)", rutkowski_cap_sims, "#dc2626"),
    ("Bowater (BLIP-2 captions)", bowater_cap_sims, "#2563eb"),
    ("Random baseline", baseline_sims, "#94a3b8"),
]:
    if data:
        fig.add_trace(go.Violin(
            y=data, name=label,
            box_visible=True, meanline_visible=True,
            line_color=color, fillcolor=color.replace(")", ", 0.25)").replace("rgb", "rgba"),
            opacity=0.7,
        ))

fig.update_layout(
    title="Figure 3e: CLIP Similarity — Original vs. Caption-Generated",
    yaxis_title="CLIP ViT-L/14 cosine similarity",
    showlegend=True,
    legend=dict(orientation="h", y=-0.15),
    font=dict(family="system-ui, sans-serif", size=12),
    violinmode="group",
)

pio.write_json(fig, os.path.join(FIGURES_DIR, "fig3e_clip_distributions.json"))
fig.write_image(os.path.join(FIGURES_DIR, "fig3e_clip_distributions.png"), scale=2)
fig.show()

In [ ]:
# Cleanup: unload SD 1.5 and CLIP before loading CSD + SD 2.1
del pipe_15
del clip_model
torch.cuda.empty_cache()
import gc; gc.collect()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")

---
## Section 4a: CSD Style Similarity

Loads CSD ViT-L from the `learn2phoenix/CSD` repo.

In [ ]:
# Load CSD model
# Checkpoint is downloaded from the CSD GitHub releases
import sys
sys.path.insert(0, os.path.join(PATH, "csd_repo"))

from huggingface_hub import hf_hub_download
import open_clip

# Download CSD checkpoint
CSD_CKPT = os.path.join(PATH, "assets", "csd_vitl.pth")
if not os.path.exists(CSD_CKPT):
    print("Downloading CSD checkpoint...")
    import urllib.request
    url = "https://github.com/learn2phoenix/CSD/releases/download/v0.1/csd_ViT-L.pth"
    urllib.request.urlretrieve(url, CSD_CKPT)
    print(f"Saved to {CSD_CKPT}")

# Load CSD
try:
    from model import CSD_CLIP
    csd_model = CSD_CLIP(name="ViT-L/14")
    ckpt = torch.load(CSD_CKPT, map_location="cpu")
    # The checkpoint may use 'model_state_dict' or 'state_dict' key
    state_dict = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
    csd_model.load_state_dict(state_dict, strict=False)
    csd_model = csd_model.to(device).eval()
    USE_CSD = True
    print("CSD model loaded.")
except Exception as e:
    print(f"CSD load failed: {e}")
    print("Falling back to CLIP ViT-L/14 as style proxy (note in results).")
    USE_CSD = False

# Load CLIP preprocessing (needed for CSD input)
_, _, csd_preprocess = open_clip.create_model_and_transforms("ViT-L-14", pretrained="openai")

if not USE_CSD:
    # Fallback: use CLIP style embedding
    clip_fallback, _, csd_preprocess = open_clip.create_model_and_transforms(
        "ViT-L-14", pretrained="openai"
    )
    clip_fallback = clip_fallback.to(device).eval()

In [ ]:
# Style embedding function

def get_style_embedding(pil_image, csd_model, preprocess, device, use_csd=True):
    """Extract style embedding using CSD (or CLIP fallback)."""
    img_t = preprocess(pil_image).unsqueeze(0).to(device)

    if use_csd:
        with torch.no_grad():
            # CSD returns (content_emb, style_emb, combined_emb)
            outputs = csd_model(img_t, img_t)  # dummy second input
            if isinstance(outputs, tuple) and len(outputs) >= 2:
                style_emb = outputs[1]
            else:
                style_emb = outputs
            style_emb = style_emb / style_emb.norm(dim=-1, keepdim=True)
        return style_emb.cpu().float()
    else:
        with torch.no_grad():
            emb = clip_fallback.encode_image(img_t)
            emb = emb / emb.norm(dim=-1, keepdim=True)
        return emb.cpu().float()

def cosine_sim(emb1, emb2):
    return (emb1 * emb2).sum().item()

# Compute prototype style vectors
print("Computing Rutkowski style prototype...")
rutkowski_style_embs = [
    get_style_embedding(item["image"], csd_model if USE_CSD else clip_fallback,
                        csd_preprocess, device, USE_CSD)
    for item in rutkowski_imgs
]
rutkowski_proto = torch.stack(rutkowski_style_embs).mean(0)
rutkowski_proto = rutkowski_proto / rutkowski_proto.norm()

print("Computing Bowater style prototype...")
bowater_style_embs = [
    get_style_embedding(item["image"], csd_model if USE_CSD else clip_fallback,
                        csd_preprocess, device, USE_CSD)
    for item in bowater_imgs
]
bowater_proto = torch.stack(bowater_style_embs).mean(0)
bowater_proto = bowater_proto / bowater_proto.norm()

print(f"Style similarity between Rutkowski and Bowater prototypes: "
      f"{cosine_sim(rutkowski_proto, bowater_proto):.3f}")

In [ ]:
# sec4_generate_sd15
# Generate images from SD 1.5 for all prompt categories

print("Loading SD 1.5 for generation...")
pipe_15 = StableDiffusionPipeline.from_pretrained(
    MODEL_ID_15, torch_dtype=torch.float16,
    safety_checker=None, requires_safety_checker=False,
).to(device)
pipe_15.enable_attention_slicing()

SEEDS_STYLE = list(range(10))  # 10 seeds for GSS
SEEDS_CONTENT = list(range(5)) # 5 seeds for content-constrained

STYLE_PROMPTS = {
    "rutkowski_general":   "A painting in the style of Greg Rutkowski",
    "rutkowski_woman":     "A painting of a woman in the style of Greg Rutkowski",
    "rutkowski_dragon":    "A painting of a dragon in the style of Greg Rutkowski",
    "rutkowski_landscape": "A landscape in the style of Greg Rutkowski",
    "bowater_general":     "A painting in the style of Charlie Bowater",
    "bowater_woman":       "A painting of a woman in the style of Charlie Bowater",
    "bowater_landscape":   "A landscape in the style of Charlie Bowater",
    "baseline_general":    "A fantasy painting",
    "baseline_woman":      "A painting of a woman",
    "baseline_landscape":  "A landscape painting",
}

def generate_batch(pipe, prompt, seeds, save_dir, prefix, n_steps=30, guidance=7.5):
    images = []
    for seed in seeds:
        gen = torch.Generator(device=device).manual_seed(seed)
        with torch.autocast("cuda"):
            img = pipe(prompt, num_inference_steps=n_steps,
                        guidance_scale=guidance, generator=gen).images[0]
        fname = f"{prefix}_seed{seed}.png"
        img.save(os.path.join(save_dir, fname))
        images.append({"image": img, "seed": seed, "path": fname, "prompt": prompt})
    return images

sd15_save_dir = os.path.join(GENERATED_DIR, "sd15_style")
sd15_generations = {}

for key, prompt in STYLE_PROMPTS.items():
    seeds = SEEDS_STYLE if "general" in key else SEEDS_CONTENT
    print(f"  Generating: {key} ({len(seeds)} images)")
    sd15_generations[key] = generate_batch(pipe_15, prompt, seeds, sd15_save_dir, f"sd15_{key}")

print("SD 1.5 generation complete.")

In [ ]:
# Compute CSD similarity for SD 1.5 generations
model_ref = csd_model if USE_CSD else clip_fallback

def compute_gss(generations, prototype, model, preprocess, device, use_csd):
    """General Style Similarity: mean CSD cosine sim between generations and prototype."""
    sims = []
    for gen in generations:
        emb = get_style_embedding(gen["image"], model, preprocess, device, use_csd)
        emb = emb / emb.norm()
        sim = cosine_sim(emb, prototype)
        sims.append(sim)
    return np.array(sims)

gss_sd15 = {}
for key, gens in sd15_generations.items():
    artist = "rutkowski" if "rutkowski" in key else ("bowater" if "bowater" in key else "baseline")
    proto  = rutkowski_proto if artist == "rutkowski" else (
             bowater_proto if artist == "bowater" else rutkowski_proto)  # baseline vs. Rutkowski proto
    gss_sd15[key] = compute_gss(gens, proto, model_ref, csd_preprocess, device, USE_CSD)
    print(f"  {key}: GSS = {gss_sd15[key].mean():.3f} ± {gss_sd15[key].std():.3f}")

# Free SD 1.5
del pipe_15
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# sec4_generate_sd21
# Load SD 2.1 and generate the same prompts

MODEL_ID_21 = "stabilityai/stable-diffusion-2-1"

print("Loading SD 2.1...")
pipe_21 = StableDiffusionPipeline.from_pretrained(
    MODEL_ID_21, torch_dtype=torch.float16,
    safety_checker=None, requires_safety_checker=False,
).to(device)
pipe_21.enable_attention_slicing()

sd21_save_dir = os.path.join(GENERATED_DIR, "sd21_style")
sd21_generations = {}

# Only run the style-relevant prompts for SD 2.1
SD21_PROMPTS = {k: v for k, v in STYLE_PROMPTS.items() if "general" in k}

for key, prompt in SD21_PROMPTS.items():
    seeds = SEEDS_STYLE
    print(f"  Generating: {key}")
    sd21_generations[key] = generate_batch(pipe_21, prompt, seeds, sd21_save_dir, f"sd21_{key}")

print("SD 2.1 generation complete.")

del pipe_21
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# Compute GSS for SD 2.1 generations
gss_sd21 = {}
for key, gens in sd21_generations.items():
    artist = "rutkowski" if "rutkowski" in key else ("bowater" if "bowater" in key else "baseline")
    proto  = rutkowski_proto if artist == "rutkowski" else (
             bowater_proto if artist == "bowater" else rutkowski_proto)
    gss_sd21[key] = compute_gss(gens, proto, model_ref, csd_preprocess, device, USE_CSD)
    print(f"  {key}: GSS = {gss_sd21[key].mean():.3f} ± {gss_sd21[key].std():.3f}")

In [ ]:
# sec4_gss_barchart
# Figure 4b: GSS bar chart comparing SD 1.5 vs SD 2.1

conditions = [
    ("Rutkowski / SD 1.5", gss_sd15.get("rutkowski_general", np.array([0])), "#dc2626"),
    ("Rutkowski / SD 2.1", gss_sd21.get("rutkowski_general", np.array([0])), "#f87171"),
    ("Bowater / SD 1.5",   gss_sd15.get("bowater_general",   np.array([0])), "#2563eb"),
    ("Baseline / SD 1.5",  gss_sd15.get("baseline_general",  np.array([0])), "#94a3b8"),
]

fig = go.Figure()

for label, gss, color in conditions:
    fig.add_trace(go.Bar(
        name=label,
        x=[label],
        y=[gss.mean()],
        error_y=dict(type="data", array=[gss.std()], visible=True),
        marker_color=color,
    ))

# Reference lines from CSD paper
fig.add_hline(y=0.85, line_dash="dot", line_color="#dc2626", opacity=0.5,
              annotation_text="CSD paper: SD 1.4 = 0.85")
fig.add_hline(y=0.48, line_dash="dot", line_color="#f87171", opacity=0.5,
              annotation_text="CSD paper: SD 2.1 = 0.48")

fig.update_layout(
    title="Figure 4b: General Style Similarity (GSS) by Condition",
    yaxis_title="GSS (CSD cosine similarity)",
    yaxis=dict(range=[0, 1]),
    showlegend=False,
    font=dict(family="system-ui, sans-serif", size=12),
)

pio.write_json(fig, os.path.join(FIGURES_DIR, "fig4b_gss_barchart.json"))
fig.write_image(os.path.join(FIGURES_DIR, "fig4b_gss_barchart.png"), scale=2)
fig.show()

# Store results
results["sec4"] = {
    "gss_rutkowski_sd15": float(gss_sd15.get("rutkowski_general", np.array([0])).mean()),
    "gss_rutkowski_sd21": float(gss_sd21.get("rutkowski_general", np.array([0])).mean()),
    "gss_bowater_sd15":   float(gss_sd15.get("bowater_general",   np.array([0])).mean()),
    "gss_baseline_sd15":  float(gss_sd15.get("baseline_general",  np.array([0])).mean()),
    "used_csd": USE_CSD,
}

In [ ]:
# sec4_gss_scatter
# Figure 4c: GSS vs CCSS scatter (following CSD paper Figure 2)

def compute_ccss(gss_dict, prefix, proto, model, preprocess, device, use_csd):
    """Content-Constrained Style Similarity: mean GSS across content-specific prompts."""
    content_keys = [k for k in gss_dict if prefix in k and "general" not in k]
    if not content_keys:
        return 0.0
    return np.mean([gss_dict[k].mean() for k in content_keys])

plot_data = [
    {
        "label": "Rutkowski / SD 1.5",
        "gss":  float(gss_sd15.get("rutkowski_general", np.array([0])).mean()),
        "ccss": compute_ccss(gss_sd15, "rutkowski", rutkowski_proto,
                              model_ref, csd_preprocess, device, USE_CSD),
        "color": "#dc2626", "symbol": "circle",
    },
    {
        "label": "Rutkowski / SD 2.1",
        "gss":  float(gss_sd21.get("rutkowski_general", np.array([0])).mean()),
        "ccss": 0.0,  # No content-constrained prompts run for SD 2.1 (keep runtime manageable)
        "color": "#f87171", "symbol": "circle-open",
    },
    {
        "label": "Bowater / SD 1.5",
        "gss":  float(gss_sd15.get("bowater_general", np.array([0])).mean()),
        "ccss": compute_ccss(gss_sd15, "bowater", bowater_proto,
                              model_ref, csd_preprocess, device, USE_CSD),
        "color": "#2563eb", "symbol": "square",
    },
    {
        "label": "Baseline / SD 1.5",
        "gss":  float(gss_sd15.get("baseline_general", np.array([0])).mean()),
        "ccss": compute_ccss(gss_sd15, "baseline", rutkowski_proto,
                              model_ref, csd_preprocess, device, USE_CSD),
        "color": "#94a3b8", "symbol": "diamond",
    },
]

fig = go.Figure()
for pt in plot_data:
    fig.add_trace(go.Scatter(
        x=[pt["gss"]], y=[pt["ccss"]],
        mode="markers+text",
        name=pt["label"],
        text=[pt["label"]],
        textposition="top center",
        marker=dict(size=14, color=pt["color"], symbol=pt["symbol"],
                    line=dict(width=2, color=pt["color"])),
    ))

fig.update_layout(
    title="Figure 4c: General vs. Content-Constrained Style Similarity",
    xaxis_title="General Style Similarity (GSS)",
    yaxis_title="Content-Constrained Style Similarity (CCSS)",
    showlegend=False,
    font=dict(family="system-ui, sans-serif", size=12),
    xaxis=dict(range=[0, 1]),
    yaxis=dict(range=[0, 1]),
)

pio.write_json(fig, os.path.join(FIGURES_DIR, "fig4c_gss_scatter.json"))
fig.write_image(os.path.join(FIGURES_DIR, "fig4c_gss_scatter.png"), scale=2)
fig.show()

In [ ]:
# sec4_gallery
# Figure 4a: 3-row gallery: originals, SD 1.5, SD 2.1

n_col = 5
fig, axes = plt.subplots(3, n_col, figsize=(15, 10))
fig.suptitle("Greg Rutkowski: Originals vs. SD 1.5 vs. SD 2.1 Generations",
              fontsize=13, fontweight="bold")

row_labels = ["Original portfolio", "SD 1.5: style prompt", "SD 2.1: style prompt"]
for row, label in enumerate(row_labels):
    axes[row][0].set_ylabel(label, fontsize=9, labelpad=8)

# Row 0: originals
for col in range(n_col):
    if col < len(rutkowski_imgs):
        axes[0][col].imshow(rutkowski_imgs[col]["image"])
    axes[0][col].axis("off")

# Row 1: SD 1.5 general prompt
sd15_general_gens = sd15_generations.get("rutkowski_general", [])[:n_col]
for col, gen in enumerate(sd15_general_gens):
    emb = get_style_embedding(gen["image"], model_ref, csd_preprocess, device, USE_CSD)
    emb = emb / emb.norm()
    sim = cosine_sim(emb, rutkowski_proto)
    axes[1][col].imshow(gen["image"])
    axes[1][col].set_xlabel(f"CSD={sim:.3f}", fontsize=8)
    axes[1][col].axis("off")

# Row 2: SD 2.1 general prompt
sd21_general_gens = sd21_generations.get("rutkowski_general", [])[:n_col]
for col, gen in enumerate(sd21_general_gens):
    emb = get_style_embedding(gen["image"], model_ref, csd_preprocess, device, USE_CSD)
    emb = emb / emb.norm()
    sim = cosine_sim(emb, rutkowski_proto)
    axes[2][col].imshow(gen["image"])
    axes[2][col].set_xlabel(f"CSD={sim:.3f}", fontsize=8)
    axes[2][col].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "fig4a_style_gallery.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: fig4a_style_gallery.png")

---
## Section 4b: CLIP vs. CSD

In [ ]:
# sec4b_clip_vs_csd
# Figure 4d: For content-constrained prompt, show top-3 by CLIP sim vs. top-3 by CSD sim

import open_clip

# Load CLIP for content retrieval
clip_model_4b, _, clip_preprocess_4b = open_clip.create_model_and_transforms(
    "ViT-L-14", pretrained="openai"
)
clip_model_4b = clip_model_4b.to(device).eval()

# Use the dragon prompt as example (content-specific)
dragon_gens = sd15_generations.get("rutkowski_dragon", [])

if dragon_gens and rutkowski_imgs:
    # Compute CLIP and CSD similarity of all generations to all portfolio images
    gen_clip_sims = []
    gen_csd_sims  = []

    for gen in dragon_gens:
        gen_clip_emb = clip_embed(gen["image"], clip_model_4b, clip_preprocess_4b, device)
        gen_csd_emb  = get_style_embedding(gen["image"], model_ref, csd_preprocess, device, USE_CSD)
        gen_csd_emb  = gen_csd_emb / gen_csd_emb.norm()

        clip_sims = [cosine_sim(gen_clip_emb, clip_embed(p["image"], clip_model_4b, clip_preprocess_4b, device))
                     for p in rutkowski_imgs]
        csd_sims  = [cosine_sim(gen_csd_emb, get_style_embedding(p["image"], model_ref, csd_preprocess, device, USE_CSD) /
                                get_style_embedding(p["image"], model_ref, csd_preprocess, device, USE_CSD).norm())
                     for p in rutkowski_imgs]
        gen_clip_sims.append(clip_sims)
        gen_csd_sims.append(csd_sims)

    # Pick the first generated image as example
    example_clip = np.array(gen_clip_sims[0])
    example_csd  = np.array(gen_csd_sims[0])

    top3_clip = np.argsort(example_clip)[::-1][:3]
    top3_csd  = np.argsort(example_csd)[::-1][:3]

    fig, axes = plt.subplots(3, 4, figsize=(14, 9))
    fig.suptitle('Figure 4d: CLIP vs. CSD\nPrompt: "A painting of a dragon in the style of Greg Rutkowski"',
                  fontsize=12, fontweight="bold")

    # Column 0: generated image
    for row in range(3):
        if row == 1:
            axes[row][0].imshow(dragon_gens[0]["image"])
            axes[row][0].set_title("Generated", fontsize=9, fontweight="bold")
        axes[row][0].axis("off")

    # Columns 1-3: CLIP top matches
    axes[0][1].set_title("Top CLIP matches\n(content retrieval)", fontsize=9, color="#2563eb")
    for col, idx in enumerate(top3_clip):
        axes[0][col + 1].imshow(rutkowski_imgs[idx]["image"])
        axes[0][col + 1].set_xlabel(f"CLIP={example_clip[idx]:.3f}", fontsize=8)
        axes[0][col + 1].axis("off")

    # Columns 1-3: CSD top matches
    axes[1][1].set_title("Top CSD matches\n(style retrieval)", fontsize=9, color="#dc2626")
    for col, idx in enumerate(top3_csd):
        axes[1][col + 1].imshow(rutkowski_imgs[idx]["image"])
        axes[1][col + 1].set_xlabel(f"CSD={example_csd[idx]:.3f}", fontsize=8)
        axes[1][col + 1].axis("off")

    # Row 2: empty for label spacing
    for col in range(4):
        axes[2][col].axis("off")

    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, "fig4d_clip_vs_csd.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: fig4d_clip_vs_csd.png")

del clip_model_4b
torch.cuda.empty_cache()

---
## Section 5: Frontier Model Analysis

**Run this section after manually collecting Gemini images.**

Prompts to run in Gemini 2.0 Flash (save 5 images per prompt):

| Prompt ID | Prompt |
|-----------|--------|
| `rutkowski_explicit` | A painting in the style of Greg Rutkowski |
| `rutkowski_descriptive` | A fantasy landscape with dramatic backlighting, rich warm colors, detailed painted environment, classical oil painting texture |
| `bowater_explicit` | A painting in the style of Charlie Bowater |
| `baseline` | A fantasy painting |

Save images to `assets/gemini/` as `{prompt_id}_{seed}.png`

In [ ]:
# sec5_analyze_gemini
# Load Gemini images and compute CSD similarity to Rutkowski prototype

gemini_prompts = [
    "rutkowski_explicit",
    "rutkowski_descriptive",
    "bowater_explicit",
    "baseline",
]

gemini_results = {}

for prompt_id in gemini_prompts:
    pattern = os.path.join(GEMINI_DIR, f"{prompt_id}_*.png")
    paths = sorted(glob.glob(pattern))
    if not paths:
        print(f"  No images found for: {prompt_id} (add to assets/gemini/)")
        gemini_results[prompt_id] = {"sims_rutkowski": [], "sims_bowater": []}
        continue

    sims_r, sims_b = [], []
    for p in paths:
        img = Image.open(p).convert("RGB").resize((512, 512), Image.LANCZOS)
        emb = get_style_embedding(img, model_ref, csd_preprocess, device, USE_CSD)
        emb = emb / emb.norm()
        sims_r.append(cosine_sim(emb, rutkowski_proto))
        sims_b.append(cosine_sim(emb, bowater_proto))

    gemini_results[prompt_id] = {
        "sims_rutkowski": sims_r,
        "sims_bowater":   sims_b,
        "gss_rutkowski":  float(np.mean(sims_r)),
        "gss_bowater":    float(np.mean(sims_b)),
    }
    print(f"  {prompt_id}: GSS(Rutkowski) = {np.mean(sims_r):.3f}")

results["sec5"] = gemini_results

In [ ]:
# Figure 5b: Grouped bar chart comparing GSS across SD 1.5, SD 2.1, Gemini

conditions_5b = [
    ("Rutkowski", [
        ("SD 1.5", results["sec4"]["gss_rutkowski_sd15"]),
        ("SD 2.1", results["sec4"]["gss_rutkowski_sd21"]),
        ("Gemini (explicit)", gemini_results.get("rutkowski_explicit", {}).get("gss_rutkowski", 0)),
        ("Gemini (descriptive)", gemini_results.get("rutkowski_descriptive", {}).get("gss_rutkowski", 0)),
    ], "#dc2626"),
    ("Bowater (control)", [
        ("SD 1.5", results["sec4"]["gss_bowater_sd15"]),
        ("SD 2.1", 0),  # Not measured
        ("Gemini (explicit)", gemini_results.get("bowater_explicit", {}).get("gss_bowater", 0)),
        ("Gemini (descriptive)", 0),
    ], "#2563eb"),
]

fig = go.Figure()

for artist, conds, color in conditions_5b:
    x_labels = [c[0] for c in conds]
    y_vals   = [c[1] for c in conds]
    fig.add_trace(go.Bar(
        name=artist,
        x=x_labels,
        y=y_vals,
        marker_color=color,
        opacity=0.85,
    ))

fig.update_layout(
    title="Figure 5b: GSS across Models and Artists",
    yaxis_title="General Style Similarity (GSS)",
    yaxis=dict(range=[0, 1]),
    barmode="group",
    legend=dict(orientation="h", y=-0.15),
    font=dict(family="system-ui, sans-serif", size=12),
)

pio.write_json(fig, os.path.join(FIGURES_DIR, "fig5b_gss_across_models.json"))
fig.write_image(os.path.join(FIGURES_DIR, "fig5b_gss_across_models.png"), scale=2)
fig.show()

---
## Section 6: Save All Results

In [ ]:
# Save all numerical results to results.json
import json

# Convert numpy types for JSON serialization
def convert(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    if isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    return obj

def deep_convert(obj):
    if isinstance(obj, dict):
        return {k: deep_convert(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [deep_convert(v) for v in obj]
    return convert(obj)

results_clean = deep_convert(results)
results_path = os.path.join(RESULTS_DIR, "results.json")

with open(results_path, "w") as f:
    json.dump(results_clean, f, indent=2)

print(f"Results saved to: {results_path}")
print(json.dumps(results_clean, indent=2))

In [ ]:
# Print experiment summary
print("=" * 60)
print("EXPERIMENT SUMMARY")
print("=" * 60)

sec2 = results.get("sec2", {})
sec3a = results.get("sec3a", {})
sec3b = results.get("sec3b", {})
sec4 = results.get("sec4", {})

print(f"\nSection 2: LAION Retrieval")
print(f"  Rutkowski hits (sim > 0.90): {sec2.get('rutkowski_hits_above_090', 'N/A')} / {len(rutkowski_imgs)}")
print(f"  Bowater hits (sim > 0.90):   {sec2.get('bowater_hits_above_090', 'N/A')} / {len(bowater_imgs)}")

print(f"\nSection 3a: Loss-Based MIA")
print(f"  Rutkowski mean loss: {sec3a.get('rutkowski_mean_loss', 'N/A'):.4f}")
print(f"  Bowater mean loss:   {sec3a.get('bowater_mean_loss', 'N/A'):.4f}")
print(f"  AUC: {sec3a.get('mia_auc', 'N/A'):.3f}")

print(f"\nSection 3b: Caption-Conditioned Generation")
print(f"  Rutkowski CLIP sim: {sec3b.get('rutkowski_clip_mean', 'N/A'):.3f}")
print(f"  Bowater CLIP sim:   {sec3b.get('bowater_clip_mean', 'N/A'):.3f}")

print(f"\nSection 4: CSD Style Similarity")
print(f"  GSS Rutkowski / SD 1.5: {sec4.get('gss_rutkowski_sd15', 'N/A'):.3f}")
print(f"  GSS Rutkowski / SD 2.1: {sec4.get('gss_rutkowski_sd21', 'N/A'):.3f}")
print(f"  GSS Bowater   / SD 1.5: {sec4.get('gss_bowater_sd15',   'N/A'):.3f}")
print(f"  Used CSD: {sec4.get('used_csd', 'N/A')}")
print()
print("All figures saved to: assets/figures/")
print("All results saved to: results/results.json")